#   SRGAN MODEL DEVELOP WITH MORE IMAGES

Since we defined some metrics in srgan2 notebook, now is time to see if we have real changes if we decide to train a srgan with all the images of the dataset, this above, to see how the model changes and if we can really see a notable diference and quantificable diference with the 3500 images model.

The first tak is to obtain a folder with all the images of the dataset getting rid of duplicates by information or name for better performance

Now we repeat the same pipeline in srgan2 but now with 12233 images.

In [20]:
import os
import shutil
import hashlib
from pathlib import Path

def get_file_hash(file_path, chunk_size=8192):
    """Calcula hash MD5 del archivo (para detectar duplicados reales)"""
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


def merge_image_folders(input_dirs, output_dir, extensions=(".jpg", ".jpeg", ".png", ".bmp", ".tif")):
    """
    Une múltiples carpetas de imágenes en una sola, evitando duplicados.

    Parámetros:
    - input_dirs: lista de rutas de carpetas
    - output_dir: carpeta destino
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    seen_names = set()
    seen_hashes = set()

    total_copied = 0
    duplicates_name = 0
    duplicates_content = 0
    total_processed = 0

    print("🔍 Iniciando proceso de unión...\n")

    for folder in input_dirs:
        folder = Path(folder)

        if not folder.exists():
            print(f"⚠️ Carpeta no encontrada: {folder}")
            continue

        print(f"📂 Procesando: {folder}")

        for file in folder.rglob("*"):
            if file.suffix.lower() not in extensions:
                continue

            total_processed += 1
            filename = file.name

            # ─── Check duplicado por nombre ───
            if filename in seen_names:
                duplicates_name += 1
                continue

            # ─── Check duplicado por contenido ───
            try:
                file_hash = get_file_hash(file)
            except Exception as e:
                print(f"❌ Error leyendo {file}: {e}")
                continue

            if file_hash in seen_hashes:
                duplicates_content += 1
                continue

            # ─── Copiar archivo ───
            dst_path = output_dir / filename

            # ⚠️ Seguridad extra: evitar overwrite raro
            if dst_path.exists():
                filename = f"{file.stem}_copy{file.suffix}"
                dst_path = output_dir / filename

            shutil.copy2(file, dst_path)

            seen_names.add(filename)
            seen_hashes.add(file_hash)
            total_copied += 1

    # ─── Resumen final ─────────────────────
    print("\n" + "="*50)
    print("📊 RESUMEN FINAL")
    print("="*50)
    print(f"Total imágenes procesadas: {total_processed}")
    print(f"Imágenes copiadas:         {total_copied}")
    print(f"Duplicados por nombre:     {duplicates_name}")
    print(f"Duplicados por contenido:  {duplicates_content}")
    print(f"Total en carpeta final:    {len(list(output_dir.glob('*')))}")
    print("="*50)

    return {
        "processed": total_processed,
        "copied": total_copied,
        "dup_name": duplicates_name,
        "dup_content": duplicates_content,
        "final_total": len(list(output_dir.glob('*')))
    }

input_dirs = [
    r"E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Aptos_messidor_dataset\Aptos_messidor_dataset\class_0",
    r"E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Aptos_messidor_dataset\Aptos_messidor_dataset\class_1",
    r"E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Test Images_new\Test Images\class_0",
    r"E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Test Images_new\Test Images\class_1"
]

output_dir = r"E:\Proyects Python based\ProyectoAvanzado2\data\processed\all_img"

merge_image_folders(input_dirs, output_dir)

🔍 Iniciando proceso de unión...

📂 Procesando: E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Aptos_messidor_dataset\Aptos_messidor_dataset\class_0
📂 Procesando: E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Aptos_messidor_dataset\Aptos_messidor_dataset\class_1
📂 Procesando: E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Test Images_new\Test Images\class_0
📂 Procesando: E:\Proyects Python based\ProyectoAvanzado2\data\raw\M1\Test Images_new\Test Images\class_1

📊 RESUMEN FINAL
Total imágenes procesadas: 25575
Imágenes copiadas:         24714
Duplicados por nombre:     0
Duplicados por contenido:  861
Total en carpeta final:    34015


{'processed': 25575,
 'copied': 24714,
 'dup_name': 0,
 'dup_content': 861,
 'final_total': 34015}

In [12]:
import os
import random
import json
import gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image, ImageFilter
import numpy as np

from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

from pathlib import Path

CHECKPOINTS_DIR = Path("E:\Proyects Python based\ProyectoAvanzado2\models\Super-resolution\srgan3")
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

Using device: cuda


In [13]:
from dataclasses import dataclass

@dataclass
class ExperimentConfig:
    name: str
    scenario: str
    downsample: str
    blur_sigma: float
    noise: str
    scale: int
    lr_size: int

EXPERIMENTS = [
    ExperimentConfig("model_1", "A", "bicubic", 1.0, "gaussian", 2, 256),
    ExperimentConfig("model_2", "A", "bicubic", 1.0, "gaussian", 4, 128),
    ExperimentConfig("model_3", "A", "bicubic", 1.0, "gaussian", 8, 64),
    ExperimentConfig("model_4", "B", "bilinear", 1.5, "gaussian", 2, 256),
    ExperimentConfig("model_5", "B", "bilinear", 1.5, "gaussian", 4, 128),
    ExperimentConfig("model_6", "B", "bilinear", 1.5, "gaussian", 8, 64),
    ExperimentConfig("model_7", "C", "lanczos", 2.0, "poisson", 2, 256),
    ExperimentConfig("model_8", "C", "lanczos", 2.0, "poisson", 4, 128),
    ExperimentConfig("model_9", "C", "lanczos", 2.0, "poisson", 8, 64),
]

In [14]:
class SRDataset(Dataset):
    def __init__(self, image_paths, cfg, patch_size=128):
        self.image_paths = image_paths
        self.cfg = cfg
        self.patch_size = patch_size

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")

        # 🔥 Crop aleatorio
        w, h = img.size
        ps = self.patch_size
        x = random.randint(0, w - ps)
        y = random.randint(0, h - ps)

        hr = img.crop((x, y, x + ps, y + ps))

        # Blur
        hr = hr.filter(ImageFilter.GaussianBlur(self.cfg.blur_sigma))

        # Downsample
        if self.cfg.downsample == "bicubic":
            lr = hr.resize((ps // self.cfg.scale, ps // self.cfg.scale), Image.BICUBIC)
        elif self.cfg.downsample == "bilinear":
            lr = hr.resize((ps // self.cfg.scale, ps // self.cfg.scale), Image.BILINEAR)
        else:
            lr = hr.resize((ps // self.cfg.scale, ps // self.cfg.scale), Image.LANCZOS)

        # Noise
        lr_np = np.array(lr).astype(np.float32)

        if self.cfg.noise == "gaussian":
            lr_np += np.random.normal(0, 5, lr_np.shape)
        else:
            lr_np = np.random.poisson(lr_np)

        lr_np = np.clip(lr_np, 0, 255)

        lr = Image.fromarray(lr_np.astype(np.uint8))

        transform = transforms.ToTensor()

        return transform(lr), transform(hr)

In [15]:
def build_loader(image_paths, cfg):
    dataset = SRDataset(image_paths, cfg)

    return DataLoader(
        dataset,
        batch_size=4,   # 🔥 seguro para tu GPU
        shuffle=True,
        num_workers=0
    )

In [16]:
class Generator(nn.Module):
    def __init__(self, scale):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 3 * (scale**2), 3, padding=1),
            nn.PixelShuffle(scale)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
def train_model(cfg, image_paths):

    print(f"\n Training {cfg.name}")

    loader = build_loader(image_paths, cfg)

    model = Generator(cfg.scale).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.L1Loss()

    for epoch in range(3):  # 🔥 empieza pequeño
        pbar = tqdm(loader)

        for lr, hr in pbar:
            lr, hr = lr.to(DEVICE), hr.to(DEVICE)

            sr = model(lr)
            loss = criterion(sr, hr)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_description(f"Loss: {loss.item():.4f}")

    return model

In [18]:
def load_images(folder):
    paths = list(Path(folder).glob("*.png"))
    print(f"Total imágenes: {len(paths)}")
    return paths

In [21]:
DATA_DIR = r"E:\Proyects Python based\ProyectoAvanzado2\data\processed\all_img"

image_paths = load_images(DATA_DIR)

# image_paths = image_paths[:500]

results = {}

for cfg in EXPERIMENTS:
    try:
        model = train_model(cfg, image_paths)

        torch.save(model.state_dict(), f"{cfg.name}.pth")

        results[cfg.name] = "ok"

        del model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print("Error:", e)
        results[cfg.name] = str(e)

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

Total imágenes: 34015

 Training model_1


Loss: 0.0113: 100%|██████████| 8504/8504 [12:46<00:00, 11.09it/s]



 Training model_2


Loss: 0.0190: 100%|██████████| 8504/8504 [14:09<00:00, 10.01it/s]



 Training model_3


...

Loss: 0.0120: 100%|██████████| 8504/8504 [13:44<00:00, 10.32it/s]



 Training model_8


Loss: 0.0213: 100%|██████████| 8504/8504 [11:33<00:00, 12.26it/s]  



 Training model_9


Loss: 0.0241: 100%|██████████| 8504/8504 [10:08<00:00, 13.98it/s]
